# Credit Risk Model — Full Evaluation

PD modeling with Logistic Regression & XGBoost, regulatory metrics, and stress testing.

In [ ]:
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_curve, auc, confusion_matrix, classification_report
from xgboost import XGBClassifier

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent if ROOT.name == 'notebooks' else ROOT
sys.path.insert(0, str(ROOT))

from src.config.paths import PROJECT_ROOT, CREDIT_CSV_FILE, require_file
from src.features.engineering import clean_raw_data, engineer_features, prepare_training_frame
from src.features.preprocessing import fit_preprocessor
from src.utils.metrics import compute_metrics
from src.models.risk_scoring import enrich_prediction, compute_ecl
from src.data.macro_loader import load_macro_scenarios
from app.api.stress.scenarios import apply_pd_shock

DATA_PATH = require_file(CREDIT_CSV_FILE)
print('Project root:', PROJECT_ROOT)

## 1. Exploratory Data Analysis

In [ ]:
df = pd.read_csv(DATA_PATH)
print('Shape:', df.shape)
print('Default rate:', f"{df['loan_status'].mean():.2%}")
df.head()

In [ ]:
df.info()
df.describe()

In [ ]:
print('Missing values:\n', df.isnull().sum())
print('Age > 100:', (df['person_age'] > 100).sum())
print('Emp length > 50:', (df['person_emp_length'] > 50).sum())
print('Max age:', df['person_age'].max())

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
sns.histplot(df['person_age'], kde=True, ax=axes[0,0]); axes[0,0].set_title('Age Distribution')
sns.histplot(df['person_income'], kde=True, ax=axes[0,1]); axes[0,1].set_title('Income Distribution')
sns.countplot(data=df, x='loan_grade', hue='loan_status', ax=axes[1,0]); axes[1,0].set_title('Default by Grade')
sns.boxplot(data=df, x='loan_status', y='loan_percent_income', ax=axes[1,1]); axes[1,1].set_title('DTI vs Default')
plt.tight_layout(); plt.show()

## 2. Data Cleaning & Feature Engineering

In [ ]:
df_clean = clean_raw_data(df)
X, y, feature_cols = prepare_training_frame(df_clean)
print('Features:', feature_cols)
print('Target distribution:\n', y.value_counts(normalize=True))

## 3. Train/Test Split & Preprocessing

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
preprocessor = fit_preprocessor(X_train)
X_train_t = preprocessor.transform(X_train)
X_test_t = preprocessor.transform(X_test)
pos_weight = (len(y_train) - y_train.sum()) / max(y_train.sum(), 1)
print('Train:', len(X_train), 'Test:', len(X_test))

## 4. Model Training — Logistic Regression & XGBoost

In [ ]:
lr = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
lr.fit(X_train_t, y_train)
lr_prob = lr.predict_proba(X_test_t)[:, 1]

xgb = XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.1, scale_pos_weight=pos_weight,
                    eval_metric='logloss', random_state=42, n_jobs=-1)
xgb.fit(X_train_t, y_train)
xgb_prob = xgb.predict_proba(X_test_t)[:, 1]
print('Models trained.')

## 5. Model Metrics

In [ ]:
lr_metrics = compute_metrics(y_test.values, lr_prob)
xgb_metrics = compute_metrics(y_test.values, xgb_prob)
metrics_df = pd.DataFrame([lr_metrics, xgb_metrics], index=['Logistic Regression', 'XGBoost'])
metrics_df

## 6. Confusion Matrix

In [ ]:
for name, prob in [('LR', lr_prob), ('XGB', xgb_prob)]:
    cm = confusion_matrix(y_test, (prob >= 0.5).astype(int))
    print(f'\n{name} Confusion Matrix:')
    print(cm)
    print(classification_report(y_test, (prob >= 0.5).astype(int)))

## 7. ROC Curve

In [ ]:
fig = go.Figure()
for name, prob, color in [('LR', lr_prob, 'blue'), ('XGB', xgb_prob, 'red')]:
    fpr, tpr, _ = roc_curve(y_test, prob)
    fig.add_trace(go.Scatter(x=fpr, y=tpr, name=f'{name} (AUC={auc(fpr,tpr):.3f})', line=dict(color=color)))
fig.add_trace(go.Scatter(x=[0,1], y=[0,1], mode='lines', name='Random', line=dict(dash='dash')))
fig.update_layout(title='ROC Curves', xaxis_title='FPR', yaxis_title='TPR')
fig.show()

## 8. Feature Importance

In [ ]:
importance = pd.Series(xgb.feature_importances_, index=feature_cols).sort_values(ascending=True).tail(15)
fig = px.bar(x=importance.values, y=importance.index, orientation='h', title='XGBoost Feature Importance (Top 15)')
fig.show()

## 9. Lift Chart

In [ ]:
def lift_chart(y_true, y_prob, n_bins=10):
    data = pd.DataFrame({'y': y_true, 'prob': y_prob})
    data['bin'] = pd.qcut(data['prob'], n_bins, duplicates='drop')
    lift = data.groupby('bin', observed=True)['y'].agg(['mean', 'count'])
    lift['lift'] = lift['mean'] / data['y'].mean()
    return lift.reset_index()

lift = lift_chart(y_test.values, xgb_prob)
fig = px.bar(lift, x='bin', y='lift', title='Lift Chart (XGBoost)', labels={'lift': 'Lift'})
fig.show()

## 10. Stress Test Examples

In [ ]:
scenarios = load_macro_scenarios()
for name, s in scenarios.items():
    print(f"{name}: portfolio PD multiplier = {s.portfolio_pd_multiplier:.2f}")

sample = df_clean.sample(200, random_state=42)
sample_probs = xgb.predict_proba(preprocessor.transform(prepare_training_frame(sample)[0]))[:, 1]

stress_rows = []
for name, scenario in scenarios.items():
    shocked = apply_pd_shock(sample_probs, scenario)
    total_ecl = 0.0
    for p, ead in zip(shocked, sample['loan_amnt']):
        stage = enrich_prediction(float(p), float(ead))['ifrs9_stage']
        total_ecl += compute_ecl(float(p), float(ead), stage)
    stress_rows.append({'scenario': name, 'avg_pd': float(shocked.mean()), 'total_ecl': total_ecl})

pd.DataFrame(stress_rows)

In [ ]:
stress_df = pd.DataFrame(stress_rows)
fig = px.bar(stress_df, x='scenario', y=['avg_pd', 'total_ecl'], barmode='group', title='Stress Test Comparison')
fig.show()